# 03d: calendar-stable features and day-1 burn-in

Tests two ideas from the ablation (see `reports/eda_summary.md`, section 5c):
1. **Stationary features.** Replace the v2 history features (7-day windows, all-time totals, account age, distinct counterparties so far, PageRank), whose values depend on how long the dataset has been running, with windows of at most 24 hours (1h / 6h / 24h) plus the gap since the last event.
2. **Burn-in.** Keep day 1 as history, but drop day-1 rows from the training rows.

Six arms (tabular, v2 history, recent; each with and without burn-in), 3 seeds each, stable XGBoost settings, threshold chosen on validation, test scored once per arm.

**Before you run:** push the latest code to GitHub (`recent_features` is new), attach the `papertrail-processed` dataset as input, turn Internet ON, attach the `DAGSHUB_TOKEN` secret, CPU only. Then **Save Version, Save & Run All (Commit)**. Expect roughly 1 to 1.5 hours. Outputs: `stationary_table.csv` and `stationary_metrics.json` in the Output tab. Download them into `PaperTrail/reports/`.


In [ ]:
!pip install -q git+https://github.com/abhibarnwal707/PaperTrail.git
import os
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
import dagshub, mlflow
dagshub.init(repo_owner="abhibarnwal707", repo_name="papertrail", mlflow=True)


In [ ]:
import glob
import json
import os
import time
from contextlib import nullcontext
from datetime import datetime
from pathlib import Path

import numpy as np
import polars as pl
import xgboost as xgb

from papertrail.features import add_out_count_24h, build_features, fit_categories
from papertrail.graph_features import HISTORY_FEATURES, RECENT_FEATURES, add_history_features, add_recent_features
from papertrail.metrics import best_f1_threshold, evaluate
from papertrail.split import assert_valid_split, split_summary, temporal_split

ON_KAGGLE = os.path.exists("/kaggle/working")
ROOT = Path(os.environ.get("PAPERTRAIL_ROOT", Path.home() / "Documents/Projects/PaperTrail"))
OUT = "/kaggle/working" if ON_KAGGLE else str(ROOT / "reports")
DATA_ROOTS = ["/kaggle/input"] if ON_KAGGLE else [str(ROOT / "data/local")]
QUICK = bool(int(os.environ.get("PAPERTRAIL_QUICK", "0")))
SEEDS = (0,) if QUICK else (0, 1, 2)

if "mlflow" in globals() and "dagshub" in globals():       # Kaggle: cell 1 of the ablation notebook already ran
    import mlflow
else:
    class _NoMlflow:
        def start_run(self, *a, **k): return nullcontext()
        def log_params(self, *a, **k): pass
        def log_metrics(self, *a, **k): pass

    mlflow = _NoMlflow()
    if os.environ.get("DAGSHUB_USER_TOKEN"):
        try:
            import dagshub
            import mlflow as _mlf
            dagshub.init(repo_owner="abhibarnwal707", repo_name="papertrail", mlflow=True)
            mlflow = _mlf
        except Exception as e:  # noqa: BLE001
            print("DagsHub logging unavailable, continuing without it:", repr(e))
print("MLflow logging:", "DagsHub" if mlflow.__class__.__name__ != "_NoMlflow" else "off", "| seeds:", SEEDS)


def find_file(name):
    for r in DATA_ROOTS:
        hits = glob.glob(f"{r}/**/{name}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found under {DATA_ROOTS}")


pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)


In [ ]:
df = pl.read_parquet(find_file("transactions.parquet")).sort(["ts", "txn_id"])
df = add_out_count_24h(df)
summary = split_summary(df)
print(summary)
assert_valid_split(summary)

t0 = time.time()
dfx = add_recent_features(add_history_features(df))
print(f"history + recent features in {time.time() - t0:.0f}s ->", dfx.shape)

train, val, test = temporal_split(dfx)
cats = fit_categories(train)


def to_matrix(part):
    base = build_features(part, cats)
    return pl.concat([base, part.select(HISTORY_FEATURES + [c for c in RECENT_FEATURES if c not in HISTORY_FEATURES])], how="horizontal").cast(pl.Float32)


Xtr_df, Xva_df, Xte_df = to_matrix(train), to_matrix(val), to_matrix(test)
ALL = Xtr_df.columns
TABULAR = [c for c in ALL if c not in HISTORY_FEATURES and c not in RECENT_FEATURES]
X_train, X_val, X_test = Xtr_df.to_numpy(), Xva_df.to_numpy(), Xte_df.to_numpy()
y_train, y_val, y_test = train["label"].to_numpy(), val["label"].to_numpy(), test["label"].to_numpy()
not_day1 = (train["ts"] >= datetime(2022, 9, 2)).to_numpy()          # burn-in mask: day-1 rows are history only
print(f"{len(TABULAR)} tabular, {len(HISTORY_FEATURES)} v2-history, {len(RECENT_FEATURES)} recent features")
print(f"burn-in drops {int((~not_day1).sum())} train rows and {int(y_train[~not_day1].sum())} of {int(y_train.sum())} positives")


In [ ]:
def smd(cols):
    out = []
    for c in cols:
        j = ALL.index(c)
        a, b = np.log1p(np.clip(X_train[:, j], 0, None)), np.log1p(np.clip(X_test[:, j], 0, None))
        sd = np.nanstd(a)
        out.append((c, float((np.nanmean(b) - np.nanmean(a)) / sd) if sd > 0 else 0.0))
    return pl.DataFrame(out, schema=["feature", "shift_sd"], orient="row")


for name, cols in (("v2 history (35)", HISTORY_FEATURES), ("recent (30)", RECENT_FEATURES)):
    t = smd(cols).with_columns(pl.col("shift_sd").abs().alias("abs"))
    print(f"\n{name}: median |shift| = {t['abs'].median():.2f} sd, max = {t['abs'].max():.2f} sd")
    print(t.sort("abs", descending=True).head(8).drop("abs"))
print("\nRead: shift_sd = how many train standard deviations the TEST mean sits above the TRAIN mean. "
      "Near 0 = same distribution. Large = the model sees values at test time it did not see in training.")


In [ ]:
KEYS = ["val_pr_auc", "test_pr_auc", "test_f1", "test_recall_at_1pct_fpr", "test_precision_at_1000"]


def run_eval(p_val, p_test):
    thr = best_f1_threshold(y_val, p_val)
    res = {"threshold": float(thr)}
    for k, v in evaluate(y_val, p_val, thr).items():
        res[f"val_{k}"] = v
    for k, v in evaluate(y_test, p_test, thr).items():
        res[f"test_{k}"] = v
    return res


def fit_multi(name, cols, burn_in=False, seeds=SEEDS):
    idx = [ALL.index(c) for c in cols]
    keep = not_day1 if burn_in else np.ones(len(y_train), dtype=bool)
    Xt, yt = X_train[keep][:, idx], y_train[keep]
    spw = float((yt == 0).sum() / yt.sum())
    cfg = dict(tree_method="hist", n_estimators=1000, learning_rate=0.05, max_depth=4, min_child_weight=10,
               reg_lambda=10.0, subsample=0.7, colsample_bytree=0.7, scale_pos_weight=float(np.sqrt(spw)),
               eval_metric="aucpr", early_stopping_rounds=50, n_jobs=-1)
    runs, t0, imp = [], time.time(), None
    for sd in seeds:
        m = xgb.XGBClassifier(**cfg, random_state=sd)
        m.fit(Xt, yt, eval_set=[(X_val[:, idx], y_val)], verbose=False)
        r = run_eval(m.predict_proba(X_val[:, idx])[:, 1], m.predict_proba(X_test[:, idx])[:, 1])
        r["best_iteration"] = int(m.best_iteration)
        runs.append(r)
        if imp is None:
            imp = dict(zip(cols, m.feature_importances_.tolist()))
    out = {"arm": name, "n_features": len(cols), "burn_in": burn_in, "train_rows": int(keep.sum()), "seeds": len(seeds),
           "seconds": round(time.time() - t0), "best_iterations": [r["best_iteration"] for r in runs]}
    for k in KEYS:
        v = [r[k] for r in runs]
        out[k + "_mean"], out[k + "_std"] = float(np.mean(v)), float(np.std(v))
    with mlflow.start_run(run_name=f"d_{name}"):
        mlflow.log_params({"arm": name, "n_features": len(cols), "burn_in": burn_in, "seeds": len(seeds)})
        mlflow.log_metrics({k: v for k, v in out.items() if k.endswith(("_mean", "_std"))})
    return out, imp


ARMS = [
    ("tabular", TABULAR, False),
    ("tabular+burnin", TABULAR, True),
    ("v2_history", TABULAR + HISTORY_FEATURES, False),
    ("v2_history+burnin", TABULAR + HISTORY_FEATURES, True),
    ("recent", TABULAR + RECENT_FEATURES, False),
    ("recent+burnin", TABULAR + RECENT_FEATURES, True),
]
rows, importances = [], {}
for name, cols, burn in ARMS:
    o, imp = fit_multi(name, cols, burn)
    rows.append(o)
    importances[name] = imp
    print(f"{name:20s} nfeat={o['n_features']:2d} rows={o['train_rows']:8d}  val={o['val_pr_auc_mean']:.3f}+-{o['val_pr_auc_std']:.3f}  "
          f"test={o['test_pr_auc_mean']:.3f}+-{o['test_pr_auc_std']:.3f}  f1={o['test_f1_mean']:.3f}  "
          f"rec@1%={o['test_recall_at_1pct_fpr_mean']:.3f}  p@1000={o['test_precision_at_1000_mean']:.3f}  "
          f"iters={o['best_iterations']}  {o['seconds']}s")


In [ ]:
tbl = pl.DataFrame(rows).drop("best_iterations")
print(tbl.select("arm", "n_features", "train_rows", "val_pr_auc_mean", "test_pr_auc_mean", "test_pr_auc_std",
                 "test_f1_mean", "test_recall_at_1pct_fpr_mean", "test_precision_at_1000_mean"))
if len(SEEDS) > 1:
    print("\nA gap between two arms is only worth discussing if it is larger than about 2x the larger test std.")
best_recent = importances.get("recent+burnin") or next(iter(importances.values()))
print("\ntop features, recent+burnin (seed 0):")
for k, v in sorted(best_recent.items(), key=lambda kv: -kv[1])[:12]:
    print(f"  {k:22s} {v:.3f}")
tbl.write_csv(f"{OUT}/stationary_table.csv")
with open(f"{OUT}/stationary_metrics.json", "w") as fh:
    json.dump({"seeds": list(SEEDS), "rows": rows, "importances": importances}, fh, indent=2, default=float)
print("Saved stationary_table.csv and stationary_metrics.json to", OUT)
